In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import torch
from torchvision.transforms import transforms as T
from torchvision.datasets import VOCDetection

dataset = VOCDetection('/datasets', download = True)

In [ ]:
dataset = VOCDetection('/datasets', image_set = 'train')

In [ ]:
dataset[0]

In [ ]:
dataset[0][0]

### Напишем функцию преобразования таргета

In [ ]:
target_names = []
for obj in dataset:
    target_names.extend([x['name'] for x in obj[1]['annotation']['object']])
target_names = list(set(target_names))
target_names

In [ ]:
name2idx = {name: i + 1 for i, name in enumerate(target_names)}
idx2name = {i + 1: name for i, name in enumerate(target_names)}

In [ ]:
def target_transform(voc_target):
    objects = voc_target['annotation']['object']
    if isinstance(objects, dict):
        objects = [objects]

    target = {}
    target['labels'] = torch.tensor(
        [name2idx[obj['name']] for obj in objects],
        dtype=torch.int64
    )
    target['boxes'] = torch.tensor(
        [
            [
                float(obj['bndbox']['xmin']),
                float(obj['bndbox']['ymin']),
                float(obj['bndbox']['xmax']),
                float(obj['bndbox']['ymax']),
            ]
            for obj in objects
        ],
        dtype=torch.float32
    )
    return target

In [ ]:
labels = target_transform(dataset[0][1])['labels']
for label in labels:
    print(idx2name[label.item()])

### Функции отрисовки прямоугольников

In [ ]:
from torchvision.utils import draw_bounding_boxes

def show(imgs):
    to_pil = T.ToPILImage()

    if not isinstance(imgs, list):
        imgs = [imgs]

    fig, axs = plt.subplots(ncols = len(imgs), squeeze = False, figsize = (16, 10))
    for i, img in enumerate(imgs):
        img = img.detach()
        img = to_pil(img)
        axs[0, i].imshow(np.asarray(img))
        axs[0, i].set(xticklabels = [], yticklabels = [], xticks = [], yticks = [])

def draw_boxes_single_image(image, boxes, labels):
    colors = ['red' for _ in boxes]
    show(draw_bounding_boxes(image, boxes, labels, colors = colors, width = 5))

In [ ]:
draw_boxes_single_image(
    T.PILToTensor()(dataset[0][0]),
    target_transform(dataset[0][1])['boxes'],
    [idx2name[label.item()] for label in target_transform(dataset[0][1])['labels']]
)

In [ ]:
for i in range(20):
    draw_boxes_single_image(
        T.PILToTensor()(dataset[i][0]),
        target_transform(dataset[i][1])['boxes'],
        [idx2name[label.item()] for label in target_transform(dataset[i][1])['labels']]
    )

### Сформируем лоадеры

In [ ]:
from torch.utils.data import DataLoader

# помещение объектов в батч (у них разные размеры таргетов, по дефолту нельзя)
def collate_fn(batch):
    return tuple(zip(*batch))

dataset_train = VOCDetection(
    '/datasets',
    image_set = 'train',
    transform = T.ToTensor(),
    target_transform = target_transform
)

dataset_test = VOCDetection(
    '/datasets',
    image_set = 'val',
    transform = T.ToTensor(),
    target_transform = target_transform
)

train_loader = DataLoader(dataset_train, batch_size = 8, shuffle = True, num_workers = 4, pin_memory = True, collate_fn = collate_fn)
test_loader = DataLoader(dataset_test, batch_size = 8, shuffle = False, num_workers = 4, pin_memory = True, collate_fn = collate_fn)

### Будем дообучать RetinaNet

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection import RetinaNet_ResNet50_FPN_Weights

def get_detection_model():
    model = retinanet_resnet50_fpn(weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT)

    num_anchors = model.head.classification_head.num_anchors
    in_channels = model.backbone.out_channels

    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_classes=len(target_names) + 1,
        num_anchors=num_anchors
    )
    return model

In [ ]:
!pip install torchmetrics

### Реализуем non maximum suppression

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import batched_nms

# возвращаем индексы прямоугольников, которые надо оставить
def apply_nms(orig_predictions, iou_thresh = 0.3):
    keep = batched_nms(orig_predictions['boxes'], orig_predictions['scores'], orig_predictions['labels'], iou_thresh)
    
    final_predictions = orig_predictions
    final_predictions['boxes'] = orig_predictions['boxes'][keep]
    final_predictions['scores'] = orig_predictions['scores'][keep]
    final_predictions['labels'] = orig_predictions['labels'][keep]
    
    return final_predictions

### Напишем функции для обучения модели и отображения результатов

In [ ]:
from torchvision.ops import batched_nms

def filter_predictions(prediction, score_thresh=0.5, iou_thresh=0.3):
    keep = prediction['scores'] > score_thresh
    prediction = {
        'boxes': prediction['boxes'][keep],
        'scores': prediction['scores'][keep],
        'labels': prediction['labels'][keep],
    }

    if len(prediction['boxes']) == 0:
        return prediction

    keep = batched_nms(
        prediction['boxes'],
        prediction['scores'],
        prediction['labels'],
        iou_thresh
    )

    return {
        'boxes': prediction['boxes'][keep],
        'scores': prediction['scores'][keep],
        'labels': prediction['labels'][keep],
    }

In [ ]:
@torch.inference_mode()
def visualize(model, batch):
    model.eval()

    to_pil = T.ToPILImage()

    xs, ys = batch
    for x, y in zip(xs, ys):
        prediction = {k: v.cpu() for k, v in model([x.to(device)])[0].items()}
        prediction = filter_predictions(prediction, score_thresh=0.5, iou_thresh=0.3)

        x_vis = (x.cpu() * 255).to(torch.uint8)

        true_boxes = draw_bounding_boxes(
            x_vis,
            boxes=y['boxes'],
            labels=[idx2name[label.item()] for label in y['labels']],
            width=3,
            colors='red'
        )

        pred_labels = [idx2name.get(label.item(), str(label.item())) for label in prediction['labels']]
        predicted_boxes = draw_bounding_boxes(
            x_vis,
            boxes=prediction['boxes'],
            labels=pred_labels,
            width=3,
            colors='red'
        )

        fig, ax = plt.subplots(1, 2, figsize=(20, 10))
        ax[0].imshow(to_pil(true_boxes))
        ax[1].imshow(to_pil(predicted_boxes))
        ax[0].axis('off')
        ax[1].axis('off')
        ax[0].set_title('True boxes')
        ax[1].set_title('Predicted boxes')
        plt.show()

In [ ]:
from tqdm import tqdm
def train(model, optimizer, loader):
    model.train()


    total_loss = 0
    for x, y in tqdm(loader, desc = 'Train'):
        x = list(_.to(device) for _ in x)
        y = [{k: v.to(device) for k, v in t.items()} for t in y]
        
        optimizer.zero_grad()
        
        output = model(x, y)
        
        loss_sum = sum(loss for loss in output.values())

        total_loss += loss_sum.item()
        
        loss_sum.backward()
        
        optimizer.step()
    print(f'Loss: {(total_loss / len(loader)):.5f}')

In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()

    metric = MeanAveragePrecision()
    
    for x, y in tqdm(loader, desc = 'Evaluation'):
        x = list(_.to(device) for _ in x)
        #y = [{k: v.to(device) for k, v in t.items()} for t in y]

        output = model(x)
        output = [{k: v.cpu() for k, v in t.items()} for t in output]

        metric.update(output, y)
    return metric.compute()['map']

In [ ]:
def plot_stats(
    train_mAp: list[float],
    test_mAp: list[float],
    title
):
    plt.figure(figsize = (16, 10))
    plt.title(title)
    plt.plot(train_mAp, label = 'Train mAP')
    plt.plot(test_mAp, label = 'Test mAP')
    plt.grid()
    plt.legend()
    plt.show()

In [ ]:
from IPython.display import clear_output
def whole_train_value_cycle(
    model,
    num_epochs: int,
    title: str
):
    batch = next(iter(test_loader))

    for epoch in range(num_epochs):
        train(model, optimizer, train_loader)

        clear_output()

        visualize(model, batch)

In [ ]:
from torch.optim import Adam

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = get_detection_model().to(device)

optimizer = Adam(model.parameters(), lr = 1e-4)

In [ ]:
whole_train_value_cycle(
    model,
    3,
    'RetinaNet finetune'
)

In [ ]:
train_mAP = evaluate(model, train_loader)
test_mAP = evaluate(model, test_loader)
print(f'Train mAP: {train_mAP:.5f}; Test mAP: {test_mAP:.5f}')